# GenAI Basics • Lesson 04: Fast-Track Image Generation and Editing APIs
**Course:** GenAI Basics • Module 2 (Multimodality)  
**Instructor:** Igor Rubanovich (`ihar_rubanovich@epam.com`)  
**Title:** Engineering Manager II & AI Ambassador @ EPAM Systems  
**Practical AI R&D:** Architect & Developer of Creator Tools — AI suite for YouTube creators (pet-project)  
**Runtime:** Google Colab (100% in-browser • Zero local installation required)

---
### 📌 Objective & Mental Model
Transition from casual web-chat image generation to **controlled, reproducible engineering pipelines**.

In this notebook, you will implement:
1. **Visual Generation Canvas (`visual_spec.json`)**: Treat image prompts as engineering contracts (sacred attributes, allowed changes, composition, camera, lighting).
2. **Official Google GenAI SDK (`google-genai`)**: Programmatic image generation and base64/binary byte decoding.
3. **Demo 1 — Text-to-Image Generation**: Generate high-fidelity product hero assets with negative space for typography.
4. **Demo 2 — Controlled Inpainting vs Vague Edit**: Compare a vague instruction ("make it nicer") with a controlled preservation prompt maintaining geometry and texture.
5. **6-Dimensional QA Scorecard**: Evaluate candidate images against professional acceptance criteria.
6. **Reproducible Homework Package**: Export `homework_l04_pack.zip` containing all artifacts for your GitHub repository.

> 💡 **No API key?** Just press **Enter** when asked for a key to run in **Zero-Cost Offline Mock Mode**.

---
## 🛠️ Step 1: Install Official Google GenAI SDK & Dependencies
Click the **Play (▶)** button on the left to install `google-genai`, `pillow`, and `matplotlib` in this cloud runtime.

In [ ]:
# Install official Google GenAI SDK, Pillow and Matplotlib (~5 seconds)
!pip install --quiet -U google-genai pillow matplotlib

---
## 🔑 Step 2: Secure API Key Configuration (Zero-Leak Protection)

### 🛡️ Option 1 (Recommended • Most Secure): Colab «Secrets» Tab (🔑 on the left)
1. Click the **Key icon (🔑)** on the left sidebar in Google Colab.
2. Click **«Add new secret»**.
3. Name: `GEMINI_API_KEY`
4. Value: Paste your key from Google AI Studio (`aistudio.google.com`).
5. Enable **«Notebook access»**.

### ⌨️ Option 2 (Interactive Input):
If no secret is found, the prompt below allows secure masked input (`getpass`).

### 🧪 Option 3 (Offline Simulation):
Just press **Enter** without a key — the notebook will execute in full simulation mode using pre-rendered benchmark assets!

In [ ]:
import getpass
import os
import json
import base64
import io
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt

raw_key = None

# 1. Safely retrieve key from Colab Secrets
try:
    from google.colab import userdata
    for k in ['GEMINI_API_KEY', 'Gemini_IPTU', 'Gemini_ITPU']:
        try:
            raw_key = userdata.get(k)
            if raw_key:
                print(f'🔒 Key securely loaded from Colab Secrets ("{k}")! Masked and protected.')
                break
        except Exception:
            pass
except Exception:
    pass

# 2. Fallback to interactive input
if not raw_key:
    raw_key = os.getenv('GEMINI_API_KEY', '').strip()

if not raw_key:
    print('⚠️ Note: No API key found in Secrets.')
    raw_key = getpass.getpass('Enter GEMINI_API_KEY (or press Enter for Zero-Cost Offline Mock Mode): ').strip()

MOCK_MODE = False
if not raw_key:
    MOCK_MODE = True
    print('🧪 MOCK MODE ACTIVE: Running 100% free offline simulation with benchmark assets.')
else:
    os.environ['GEMINI_API_KEY'] = raw_key
    print('✅ LIVE MODE ACTIVE: Connected to Google Cloud GenAI API.')

# Setup output directory
OUT_DIR = Path('out')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'📁 Output directory ready: {OUT_DIR.resolve()}')

---
## 📐 Step 3: The Visual Generation Canvas (`visual_spec.json`)

In professional engineering pipelines, we do not throw stream-of-consciousness prompts at models.
We treat prompt construction as an **engineering specification**:
- **Subject**: Exact object identity and physical features.
- **Environment**: Supporting surface, room, architectural setting.
- **Composition**: Negative space for typography, camera placement, rule of thirds.
- **Lighting**: Temperature, directionality, contrast, shadow sharpness.
- **Constraints (Negative Prompts)**: Strict boundary conditions (no text, no extra handles, no duplicates).
- **Output Format**: Target aspect ratio and resolution.

Let's define `visual_spec.json` and compile it into an operational prompt.

In [ ]:
visual_spec = {
    "subject": "speckled ceramic coffee mug; exact cylindrical shape preserved",
    "environment": "warm modern studio with pale travertine surface and soft architectural foliage in background",
    "composition": "single hero object on the left; clean negative space on the right for deterministic typography",
    "camera": "eye-level product photography, 85mm lens, shallow depth of field",
    "lighting": "warm late-afternoon side light; soft diffused natural shadow; balanced specular highlights",
    "style": "premium natural editorial commercial photography; not a 3D render, not an illustration",
    "constraints": [
        "single mug only",
        "no text or typography",
        "no watermark or logos",
        "no extra handle",
        "preserve authentic proportions and ceramic glaze"
    ],
    "output": "16:9, 2K"
}

# Save spec to disk
with open('visual_spec.json', 'w', encoding='utf-8') as f:
    json.dump(visual_spec, f, indent=2, ensure_ascii=False)

# Deterministic Prompt Builder (Contract compilation)
structured_prompt = " | ".join([
    f"Subject: {visual_spec['subject']}",
    f"Environment: {visual_spec['environment']}",
    f"Composition: {visual_spec['composition']}",
    f"Camera: {visual_spec['camera']}",
    f"Lighting: {visual_spec['lighting']}",
    f"Style: {visual_spec['style']}",
    "Constraints: " + "; ".join(visual_spec['constraints']),
    f"Output: {visual_spec['output']}",
])

print('📝 COMPILED STRUCTURED PROMPT:')
print('-' * 80)
print(structured_prompt)
print('-' * 80)

---
## 🎨 Step 4: Demo 1 — Programmatic Text-to-Image Generation

Key difference from text LLM APIs:
- Text LLM APIs stream incremental Unicode tokens.
- Image APIs return an **atomic binary payload** (Base64 encoded string or raw PNG/JPEG bytes).
- Your application code must decode the Base64 data and store it in an object store (S3, GCS) or disk.

Let's execute the call and inspect the generated image:

In [ ]:
hero_img_path = OUT_DIR / 'generated_product_hero.png'

if not MOCK_MODE:
    try:
        from google import genai
        client = genai.Client()
        print('📡 Sending request to Google GenAI Image API (imagen-3.0-generate-002)...')
        
        # Call official Google GenAI Imagen model
        result = client.models.generate_images(
            model='imagen-3.0-generate-002',
            prompt=structured_prompt,
            config=dict(
                number_of_images=1,
                aspect_ratio='16:9',
                output_mime_type='image/png'
            )
        )
        raw_bytes = result.generated_images[0].image.image_bytes
        hero_img_path.write_bytes(raw_bytes)
        print(f'✅ Image successfully generated and saved to {hero_img_path} ({len(raw_bytes):,} bytes)')
    except Exception as e:
        print(f'⚠️ Live API call failed ({e}). Falling back to benchmark mock visualization.')
        MOCK_MODE = True

if MOCK_MODE:
    # Create a high-quality benchmark mock image for demonstration
    mock_img = Image.new('RGB', (1280, 720), color=(26, 32, 44))
    draw = ImageDraw.Draw(mock_img)
    # Draw travertine table
    draw.rectangle([0, 420, 1280, 720], fill=(215, 204, 186))
    # Draw mug shadow
    draw.ellipse([340, 520, 560, 580], fill=(160, 150, 135))
    # Draw ceramic mug body
    draw.rounded_rectangle([360, 320, 520, 550], radius=18, fill=(245, 240, 230), outline=(120, 110, 95), width=3)
    # Draw handle
    draw.arc([490, 360, 565, 510], start=-90, end=90, fill=(245, 240, 230), width=16)
    # Draw negative space guide
    draw.rectangle([680, 100, 1220, 620], outline=(56, 189, 248), width=2)
    draw.text((700, 120), 'SAFE NEGATIVE SPACE FOR TYPOGRAPHY', fill=(56, 189, 248))
    draw.text((700, 150), 'Single Hero Mug on Left (Eye-Level 85mm)', fill=(148, 163, 184))
    draw.text((700, 180), 'Travertine Studio Surface • Warm Side Lighting', fill=(148, 163, 184))
    draw.text((700, 210), 'Aspect Ratio: 16:9 • Clean Geometry Preserved', fill=(52, 211, 153))
    mock_img.save(hero_img_path)
    print(f'✅ Mock benchmark image generated: {hero_img_path}')

# Display image inline
display_img = Image.open(hero_img_path)
plt.figure(figsize=(12, 6.75))
plt.imshow(display_img)
plt.axis('off')
plt.title('Demo 1 Output: Generated Product Hero (16:9 Aspect Ratio)', fontsize=14, pad=12)
plt.tight_layout()
plt.show()

---
## 🔄 Step 5: Demo 2 — Vague Edit vs Controlled Inpainting

Here is the central lesson of production image engineering:
- **Vague Prompt ("Make this mug look premium in a studio")**:
  The model drifts! The ceramic glaze changes, the handle warps, the proportions skew, or an extra mug appears.
- **Controlled Edit with Preservation Rules**:
  We explicitly state what is **Sacred** (mug geometry, speckles, handle, camera angle) and what is **Allowed** (background travertine, lighting, soft shadow).

Let's compare them side-by-side:

In [ ]:
vague_prompt = 'Make this mug photo look like a premium campaign image in a beautiful warm studio.'
controlled_prompt = (
    'Replace only the background and supporting surface with a warm sunlit travertine studio. '
    'PRESERVE EXACT: mug shape, speckled ceramic glaze, handle position, proportions and camera angle. '
    'Single mug only. No duplicate handles, no text, no watermark.'
)

vague_path = OUT_DIR / 'edit_vague.png'
controlled_path = OUT_DIR / 'edit_controlled.png'

# Benchmark comparison visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Candidate 1: Vague
img_vague = Image.new('RGB', (640, 480), color=(45, 30, 25))
d_v = ImageDraw.Draw(img_vague)
d_v.rectangle([0, 320, 640, 480], fill=(130, 90, 70))
# Distorted mug with geometry drift
d_v.rounded_rectangle([200, 180, 410, 400], radius=35, fill=(210, 160, 140), outline=(239, 68, 68), width=4)
d_v.arc([360, 220, 460, 340], start=-90, end=90, fill=(210, 160, 140), width=24)
d_v.text((30, 30), 'FAIL: Geometry & Texture Drift', fill=(239, 68, 68))
d_v.text((30, 60), '• Handle warped and overly thick', fill=(255, 255, 255))
d_v.text((30, 85), '• Glaze texture lost (wrong material)', fill=(255, 255, 255))
d_v.text((30, 110), '• Proportions deformed', fill=(255, 255, 255))
img_vague.save(vague_path)

# Candidate 2: Controlled
img_ctrl = Image.new('RGB', (640, 480), color=(26, 32, 44))
d_c = ImageDraw.Draw(img_ctrl)
d_c.rectangle([0, 320, 640, 480], fill=(215, 204, 186))
d_c.ellipse([210, 380, 420, 430], fill=(160, 150, 135))
# Preserved mug body
d_c.rounded_rectangle([230, 180, 390, 410], radius=16, fill=(245, 240, 230), outline=(52, 211, 153), width=4)
d_c.arc([360, 220, 435, 370], start=-90, end=90, fill=(245, 240, 230), width=14)
d_c.text((30, 30), 'PASS: Sacred Attributes Preserved', fill=(52, 211, 153))
d_c.text((30, 60), '• Ceramic glaze and speckles intact', fill=(255, 255, 255))
d_c.text((30, 85), '• Handle geometry and position identical', fill=(255, 255, 255))
d_c.text((30, 110), '• Travertine studio background replaced', fill=(255, 255, 255))
img_ctrl.save(controlled_path)

axes[0].imshow(img_vague)
axes[0].set_title('❌ Vague Prompt (Hallucinated Geometry)', fontsize=12, color='red', pad=8)
axes[0].axis('off')

axes[1].imshow(img_ctrl)
axes[1].set_title('✅ Controlled Inpainting (Contract Enforced)', fontsize=12, color='green', pad=8)
axes[1].axis('off')

plt.tight_layout()
plt.show()

---
## 📊 Step 6: The 6-Dimensional QA Scorecard

Never evaluate production AI images with "looks cool" or subjective opinions.
Use our standardized 6-dimensional rubric (Score 0-2 for each criterion):

| Dimension | 0 (Fail) | 1 (Partial) | 2 (Pass) |
| :--- | :--- | :--- | :--- |
| **1. Instruction Fidelity** | Misses core task | Partially accurate | Matches visual contract |
| **2. Attribute Preservation** | Critical drift/hallucination | Minor drift | Sacred attributes 100% preserved |
| **3. Composition / Safe Space** | Unusable crop/ratio | Usable with manual fix | Presentation & channel-ready |
| **4. Structural Artifacts** | Extra limbs, distorted handles | Minor polish required | Clean at 100% review zoom |
| **5. Text / Watermark Hygiene** | Unwanted AI gibberish | Tiny watermark | Intentionally absent or exact |
| **6. Channel Delivery** | Wrong format/dimensions | Re-encoding needed | Ready for immediate deployment |

Let's compute the programmatic scorecard:

In [ ]:
import csv

qa_data = [
    ["Candidate", "Instruction_Fidelity", "Attribute_Preservation", "Composition_Safe_Space", "Structural_Artifacts", "Text_Watermarks", "Channel_Delivery", "Total_Score_12", "Verdict"],
    ["Candidate A (Vague Prompt)", 1, 0, 1, 1, 2, 2, 7, "REJECT (Geometry Drift)"],
    ["Candidate B (Controlled Inpainting)", 2, 2, 2, 2, 2, 2, 12, "ACCEPT (Production Ready)"]
]

csv_path = OUT_DIR / 'qa_scorecard.csv'
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerows(qa_data)

print(f'📊 QA Scorecard saved to: {csv_path}')
print('-' * 90)
for row in qa_data:
    print(f'{row[0]:<35} | Total: {row[7]}/12 | Verdict: {row[8]}')
print('-' * 90)

---
## 📦 Step 7: Export Reproducible Homework Package

Pack all your deliverables into a clean submission zip archive ready for your GitHub homework repo:
- `brief.md` (Visual Generation Canvas)
- `visual_spec.json` (Structured specification)
- `prompt_v1.txt` & `prompt_v2.txt` (Initial & revised prompts)
- `candidates/` (Output images)
- `qa_scorecard.csv` (Rubric evaluations)
- `reflection.md` (Engineering rationale)

In [ ]:
import zipfile

# 1. Create homework folder structure
hw_dir = Path('homework_l04_pack')
hw_dir.mkdir(exist_ok=True)
(hw_dir / 'candidates').mkdir(exist_ok=True)

# 2. Copy spec & create briefs
(hw_dir / 'visual_spec.json').write_text(Path('visual_spec.json').read_text(encoding='utf-8'), encoding='utf-8')
(hw_dir / 'qa_scorecard.csv').write_text(Path('out/qa_scorecard.csv').read_text(encoding='utf-8'), encoding='utf-8')
(hw_dir / 'prompt_v1.txt').write_text(vague_prompt, encoding='utf-8')
(hw_dir / 'prompt_v2.txt').write_text(controlled_prompt, encoding='utf-8')

brief_content = """# Visual Generation Brief: E-Commerce Product Hero
**Student:** GenAI Basics Participant
**Course:** GenAI Basics • Lesson 04
**Business Outcome:** Premium e-commerce hero image for seasonal catalogue.
**Sacred Attributes:** Speckled ceramic mug geometry, handle placement, glaze texture.
**Allowed Changes:** Background surface (pale travertine), warm afternoon studio lighting.
**Composition:** Negative space on right for campaign headline and pricing.
"""
(hw_dir / 'brief.md').write_text(brief_content, encoding='utf-8')

reflection_content = """# Engineering Reflection
1. Text prompts alone are insufficient to guarantee exact geometry preservation across model calls.
2. Inpainting with mask boundaries and reference image conditioning provides significantly tighter control than text-only regeneration.
3. Prompt engineering for images must function as an operational specification (Subject, Environment, Composition, Lighting, Constraints) rather than subjective adjectives.
4. Production image pipelines require the 10/90 architecture: schema validation, S3 storage, Base64 decoding, metadata logging, and QA gates.
"""
(hw_dir / 'reflection.md').write_text(reflection_content, encoding='utf-8')

# Copy candidate images
for src, name in [(hero_img_path, 'hero.png'), (vague_path, 'vague.png'), (controlled_path, 'controlled.png')]:
    if src.exists():
        (hw_dir / 'candidates' / name).write_bytes(src.read_bytes())

# 3. Create ZIP archive
zip_path = Path('homework_l04_pack.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file in hw_dir.rglob('*'):
        if file.is_file():
            zipf.write(file, arcname=file.relative_to(hw_dir.parent))

print(f'🎉 Homework package created: {zip_path.resolve()} ({zip_path.stat().st_size:,} bytes)')
print('👉 Download homework_l04_pack.zip and commit to your genai-homeworks/L04 repo!')